# 01. Source Data Ingestion

## Banking Transaction Lakehouse

**Purpose:** Generate and inspect banking transaction data before it enters the Bronze layer.

### Scope

* Create a controlled sample transaction dataset.
* Generate larger synthetic datasets using Spark-native operations.
* Inspect schema, record counts and data quality scenarios.
* Prepare source data for downstream pipeline processing.

**Implementation:** Reusable data generation functions are maintained in `src/ingestion/`.


In [0]:
from src.ingestion.sample_source import create_sample_transactions
from src.ingestion.synthetic_data_generator import generate_synthetic_transactions

print("Source ingestion notebook initialized.")

In [0]:
# Create controlled sample transaction data
sample_df = create_sample_transactions(spark)

# Inspect schema
sample_df.printSchema()

# Display source records
display(sample_df)

# Verify total records
print("Total source records:", sample_df.count())

In [0]:
# Generate 10,000 synthetic banking transactions
synthetic_df = generate_synthetic_transactions(
    spark,
    record_count=10000
)

# Inspect schema and sample records
synthetic_df.printSchema()
display(synthetic_df.limit(20))

print("Total synthetic records:", synthetic_df.count())

In [0]:
from pyspark.sql import functions as F

quality_summary = synthetic_df.agg(
    F.count("*").alias("total_records"),

    F.sum(
        F.when(F.col("customer_id").isNull(), 1).otherwise(0)
    ).alias("missing_customer_id"),

    F.sum(
        F.when(F.col("amount") < 0, 1).otherwise(0)
    ).alias("negative_amount"),

    F.sum(
        F.when(
            ~F.col("transaction_type").isin("DEBIT", "CREDIT"),
            1
        ).otherwise(0)
    ).alias("invalid_transaction_type")
)

display(quality_summary)

In [0]:
# Profile synthetic transaction data

synthetic_df.select(
    F.count("*").alias("total_transactions"),
    F.countDistinct("transaction_id").alias("unique_transactions"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.countDistinct("account_id").alias("unique_accounts"),
    F.min("transaction_date").alias("first_transaction_date"),
    F.max("transaction_date").alias("last_transaction_date"),
    F.round(F.avg("amount"), 2).alias("average_amount"),
    F.min("amount").alias("minimum_amount"),
    F.max("amount").alias("maximum_amount")
).display()